# Module 12 - Threading Drills

All exercises use files in the `data/` directory (10 server log files).

---

### Instructions
1. Read each exercise carefully
2. Write your solution in the blank code cell below it
3. Run the **Testing** section when done
4. All tests green = ready for the next module

---

## Part 1 — Basic Threads (Exercises 1-3)


### Exercise 1

Using `Thread(target=..., args=(...))`, write a function `count_events(hostname)`
that opens `data/{hostname}.log` and counts total non-empty lines.
Return nothing — just print: `[hostname] N events`.

Discover all `.log` files dynamically with `os.listdir('data')`.
Launch one thread per host. Join all threads.

Assign `total_hosts_scanned` = the number of threads launched.


In [ ]:
# Exercise 1
import os
from threading import Thread


### Exercise 2

Subclass `Thread` to create `HostLogThread` with:
- `__init__(self, hostname)` storing `hostname`
- `self.line_count = 0` and `self.event_types = set()` as instance attributes
- `run()` that reads the log, counts lines, and collects unique event type strings
  (4th `|`-separated field on each line)

Launch for all 10 hosts. After joining, assign:
- `total_lines` = sum of all `line_count` values
- `unique_event_types` = union of all `event_types` sets


In [ ]:
# Exercise 2
import os
from threading import Thread


### Exercise 3

Demonstrate the difference between `start()` and `run()`.

Write `slow_reader(hostname)` that sleeps 0.3 seconds then prints the host name.

Time how long it takes to run 5 threads using:
- `t.run()` for each (sequential — **wrong** way)
- `t.start()` + `t.join()` for each (parallel — **correct** way)

Assign:
- `sequential_time` = elapsed seconds using `.run()`
- `parallel_time` = elapsed seconds using `.start()` + `.join()`
- `is_parallel_faster` = `True` if `parallel_time < sequential_time`


In [ ]:
# Exercise 3
import time
from threading import Thread


---

## Part 2 — `join()` and Result Collection (Exercises 4-6)


### Exercise 4

Using `HostLogThread` from Exercise 2 (or a new class), collect the hostname
of the host with the **most** `INTRUSION_ATTEMPT` events.

Store the result as:
- `most_attacked_host` — the hostname string
- `max_intrusion_count` — the count (int)

Ensure you join all threads before reading results.


In [ ]:
# Exercise 4
import os
from threading import Thread


### Exercise 5

Write a threaded function that reads all 10 logs and builds a list `all_events`
of dicts: `{"hostname": ..., "timestamp": ..., "source_ip": ..., "event_type": ..., "detail": ...}`

Parse each pipe-delimited line. Use `Thread(target=..., args=(...))` and a shared list.
Use a `Lock` to protect appends to `all_events`.

After joining, assign:
- `total_parsed_events` = `len(all_events)`


In [ ]:
# Exercise 5
import os
from threading import Thread, Lock


### Exercise 6

Using threads, read all 10 log files and collect **only the lines where**
`SSH_LOGIN_SUCCESS` immediately follows one or more `SSH_LOGIN_FAILED` from the same IP
**within the same host's log** (line-by-line within each file).

Assign:
- `suspicious_successes` — a list of `(hostname, line)` tuples
- `suspicious_count` = `len(suspicious_successes)`


In [ ]:
# Exercise 6
import os
from threading import Thread, Lock


---

## Part 3 — Locks (Exercises 7-9)


### Exercise 7

Demonstrate a race condition — then fix it with a `Lock`.

Write `increment_without_lock()` — 10 threads each increment a shared `unsafe_counter`
by 1 in a tight loop of 1000 iterations. Print the final value.

Write `increment_with_lock()` — same, but protected by a `Lock`.

Assign:
- `locked_result` = the result of `increment_with_lock()` — must be 10000
- `lock_is_correct` = `True` if `locked_result == 10000`


In [ ]:
# Exercise 7
from threading import Thread, Lock


### Exercise 8

Using a `Lock` and threads, write a log aggregator that reads all 10 host logs
and builds a shared dict `events_per_host: {hostname: total_event_count}`.

Rules:
- One thread per host
- Use a `Lock` to protect writes to the shared dict
- Collect counts locally per thread, write to dict once at the end of each thread

After joining, assign `events_per_host` and `host_with_fewest_events` (hostname with min count).


In [ ]:
# Exercise 8
import os
from threading import Thread, Lock


### Exercise 9

Using `RLock` and the subclassing approach, write a `ThreatReportWriter` class that:
- Takes `hostname` and a shared `RLock`
- Reads `data/{hostname}.log`
- Collects all lines where `source_ip` is `185.220.101.47` (known attacker)
- Appends them to `data/attacker_activity.txt`, holding the lock for the write
  (include a `--- hostname ---` separator before each host's block)

Launch for all 10 hosts. After joining, count total lines in `data/attacker_activity.txt`
(excluding separator lines) and assign to `attacker_line_count`.


In [ ]:
# Exercise 9
import os
from threading import Thread, RLock


---

## Part 4 — `ThreadPoolExecutor` (Exercises 10-12)


### Exercise 10

Using `ThreadPoolExecutor.map()`, write a function `get_unique_ips(hostname)` that
returns the set of all unique source IPs seen in that host's log.

Collect results across all 10 hosts and compute:
- `infrastructure_unique_ips` — the union of all IP sets
- `infrastructure_ip_count` = `len(infrastructure_unique_ips)`


In [ ]:
# Exercise 10
import os
from concurrent.futures import ThreadPoolExecutor


### Exercise 11

Using `submit()` and `as_completed()`, process all 10 hosts.
Each call to `analyze_host(hostname)` should return a dict:
`{"hostname": ..., "risk_score": ...}` where `risk_score` is computed as:

```
risk_score = (failed_logins * 1) + (intrusions * 5) + (brute_force_wins * 10)
```

Print results as they arrive (not in input order — as_completed).
Assign `highest_risk_host` = the hostname with the maximum `risk_score`.


In [ ]:
# Exercise 11
import os
from concurrent.futures import ThreadPoolExecutor, as_completed


### Exercise 12

Using `ThreadPoolExecutor`, write a function `count_by_event_type(hostname)` that
returns a dict: `{event_type: count}` for all event types in that host's log.

Then use `executor.map()` to get per-host dicts, and merge them into a single
infrastructure-wide `event_type_totals: {event_type: total_count}` dict.

Assign `most_common_event` = the event type with the highest total count.


In [ ]:
# Exercise 12
import os
from concurrent.futures import ThreadPoolExecutor


---

## Part 5 — Putting It Together (Exercises 13-15)


### Exercise 13

Using a `queue.Queue`, build a producer-consumer pipeline:
- **Producers** (10 threads): each reads a host log and puts
  `(hostname, source_ip, event_type)` tuples onto the queue for every event
- **Consumer** (1 thread): reads from the queue and builds
  `ip_activity: {source_ip: {event_type: count}}`

After the pipeline completes, assign:
- `ip_activity` = the full dict
- `most_active_ip` = IP with the highest total event count across all event types


In [ ]:
# Exercise 13
import os, queue
from threading import Thread


### Exercise 14

Write a threaded incident triage tool:

1. Use `ThreadPoolExecutor` to ingest all 10 host logs in parallel
2. Each host log returns a dict with: `hostname`, `failed_logins`, `intrusions`,
   `brute_force_wins`, `unique_attacker_ips` (set)
3. After all complete, compute a `triage_report`:
   - `total_hosts`: count
   - `hosts_with_intrusions`: list of hostnames where `intrusions > 0`
   - `hosts_compromised`: list of hostnames where `brute_force_wins > 0`
   - `all_attacker_ips`: sorted list of unique attacker IPs across all hosts
4. Write `data/triage_report.json` with `indent=2`

Assign `compromised_count` = `len(triage_report['hosts_compromised'])`


In [ ]:
# Exercise 14
import os, json
from concurrent.futures import ThreadPoolExecutor


### Exercise 15 — Full Threaded SIEM Pipeline

Build a complete multi-threaded log processing pipeline:

1. **Ingestion** — `ThreadPoolExecutor`: read all 10 host logs in parallel,
   parse every line into a structured dict
2. **Detection** — for each event, apply these rules:
   - Rule A: `SSH_LOGIN_FAILED` from same IP 3+ times on any host → `BRUTE_FORCE_ALERT`
   - Rule B: `INTRUSION_ATTEMPT` on any host → `INTRUSION_ALERT`
   - Rule C: `Brute force succeeded` → `COMPROMISE_ALERT` (highest severity)
3. **Output** — write two files:
   - `data/alerts_generated.csv` with columns: `alert_type`, `hostname`, `source_ip`, `timestamp`, `detail`
   - `data/siem_summary.json` with total counts per alert type and list of compromised hosts

Assign:
- `total_alerts_generated` = total rows in the CSV (excluding header)
- `compromise_alerts` = count of `COMPROMISE_ALERT` rows


In [ ]:
# Exercise 15
import os, csv, json
from concurrent.futures import ThreadPoolExecutor
from threading import Lock


---

## Testing

Run the cell below. **Don't worry about the `class Test...` syntax** — `unittest` is covered later.


In [ ]:
import unittest, os, json, csv

class TestThreadingDrills(unittest.TestCase):

    # ---- Part 1 ----

    def test_01_total_hosts_scanned(self):
        self.assertEqual(total_hosts_scanned, 10,
            "total_hosts_scanned must be 10 — one thread per log file")

    def test_02_subclass_thread(self):
        self.assertIsInstance(total_lines, int,
            "total_lines must be an int")
        self.assertGreater(total_lines, 0,
            "total_lines must be > 0 — threads must have read the files")
        self.assertIsInstance(unique_event_types, set,
            "unique_event_types must be a set")
        self.assertGreater(len(unique_event_types), 1,
            "unique_event_types must contain multiple event types")

    def test_03_start_vs_run(self):
        self.assertIsInstance(sequential_time, float,
            "sequential_time must be a float")
        self.assertIsInstance(parallel_time, float,
            "parallel_time must be a float")
        self.assertTrue(is_parallel_faster,
            "Parallel execution must be faster than sequential for IO-bound tasks")
        self.assertGreater(sequential_time, parallel_time,
            "sequential_time must be greater than parallel_time")

    # ---- Part 2 ----

    def test_04_most_attacked_host(self):
        self.assertIsInstance(most_attacked_host, str,
            "most_attacked_host must be a hostname string")
        self.assertIsInstance(max_intrusion_count, int,
            "max_intrusion_count must be an int")
        self.assertGreater(max_intrusion_count, 0,
            "max_intrusion_count must be > 0")

    def test_05_all_events(self):
        self.assertIsInstance(total_parsed_events, int,
            "total_parsed_events must be an int")
        self.assertGreater(total_parsed_events, 100,
            "10 hosts x 12 events = 120 total events expected")
        self.assertEqual(total_parsed_events, len(all_events),
            "total_parsed_events must equal len(all_events)")
        if all_events:
            sample = all_events[0]
            for key in ["hostname", "timestamp", "source_ip", "event_type", "detail"]:
                self.assertIn(key, sample,
                    f"Each event dict must have key '{key}'")

    def test_06_suspicious_successes(self):
        self.assertIsInstance(suspicious_successes, list,
            "suspicious_successes must be a list")
        self.assertEqual(suspicious_count, len(suspicious_successes),
            "suspicious_count must equal len(suspicious_successes)")
        self.assertGreater(suspicious_count, 0,
            "There are brute-force successes in the test data — should find at least one")

    # ---- Part 3 ----

    def test_07_lock_correctness(self):
        self.assertEqual(locked_result, 10000,
            "locked_result must be exactly 10000 — Lock prevents race condition")
        self.assertTrue(lock_is_correct,
            "lock_is_correct must be True")

    def test_08_events_per_host(self):
        self.assertIsInstance(events_per_host, dict,
            "events_per_host must be a dict")
        self.assertEqual(len(events_per_host), 10,
            "events_per_host must have 10 entries — one per host")
        self.assertIsInstance(host_with_fewest_events, str,
            "host_with_fewest_events must be a hostname string")

    def test_09_attacker_activity_file(self):
        self.assertTrue(os.path.exists("data/attacker_activity.txt"),
            "data/attacker_activity.txt must be created")
        self.assertIsInstance(attacker_line_count, int,
            "attacker_line_count must be an int")
        self.assertGreater(attacker_line_count, 0,
            "attacker_line_count must be > 0 — attacker 185.220.101.47 appears in all logs")

    # ---- Part 4 ----

    def test_10_infrastructure_ips(self):
        self.assertIsInstance(infrastructure_unique_ips, set,
            "infrastructure_unique_ips must be a set")
        self.assertGreater(infrastructure_ip_count, 0,
            "infrastructure_ip_count must be > 0")
        self.assertEqual(infrastructure_ip_count, len(infrastructure_unique_ips),
            "infrastructure_ip_count must equal len(infrastructure_unique_ips)")

    def test_11_highest_risk_host(self):
        self.assertIsInstance(highest_risk_host, str,
            "highest_risk_host must be a hostname string")
        self.assertTrue(highest_risk_host.endswith(".log") is False,
            "highest_risk_host must be a hostname, not a filename (no .log extension)")

    def test_12_event_type_totals(self):
        self.assertIsInstance(event_type_totals, dict,
            "event_type_totals must be a dict")
        self.assertGreater(len(event_type_totals), 1,
            "event_type_totals must have multiple event types")
        self.assertIsInstance(most_common_event, str,
            "most_common_event must be a string (event type name)")

    # ---- Part 5 ----

    def test_13_ip_activity(self):
        self.assertIsInstance(ip_activity, dict,
            "ip_activity must be a dict")
        self.assertIsInstance(most_active_ip, str,
            "most_active_ip must be a string (IP address)")
        self.assertGreater(len(ip_activity), 0,
            "ip_activity must not be empty")

    def test_14_triage_report(self):
        self.assertTrue(os.path.exists("data/triage_report.json"),
            "data/triage_report.json must be created")
        with open("data/triage_report.json", "r") as f:
            report = json.load(f)
        for key in ["total_hosts", "hosts_with_intrusions", "hosts_compromised", "all_attacker_ips"]:
            self.assertIn(key, report, f"triage_report must contain key '{key}'")
        self.assertEqual(report["total_hosts"], 10,
            "total_hosts must be 10")
        self.assertIsInstance(compromised_count, int,
            "compromised_count must be an int")
        self.assertEqual(compromised_count, len(report["hosts_compromised"]),
            "compromised_count must equal len(hosts_compromised) in the report")

    def test_15_siem_pipeline(self):
        self.assertGreater(total_alerts_generated, 0,
            "total_alerts_generated must be > 0")
        self.assertGreaterEqual(compromise_alerts, 0,
            "compromise_alerts must be >= 0")
        self.assertTrue(os.path.exists("data/alerts_generated.csv"),
            "data/alerts_generated.csv must be created")
        self.assertTrue(os.path.exists("data/siem_summary.json"),
            "data/siem_summary.json must be created")
        with open("data/alerts_generated.csv", newline="") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
        self.assertEqual(len(rows), total_alerts_generated,
            "total_alerts_generated must equal row count in alerts_generated.csv")
        for row in rows:
            self.assertIn("alert_type", row,
                "Each CSV row must have 'alert_type' column")
            self.assertIn(row["alert_type"],
                ["BRUTE_FORCE_ALERT", "INTRUSION_ALERT", "COMPROMISE_ALERT"],
                f"Unexpected alert_type: {row['alert_type']}")


unittest.main(argv=[""], verbosity=2, exit=False)


---

## Well Done!

If all 15 tests pass, you understand how to write thread-safe, parallel security tools —
a skill that separates basic scripters from real security engineers.

**Next module:** Decorators — wrapping functions to add logging, timing, and
access control without modifying the original code.
